In [1]:
import pandas as pd
import numpy as np
import sklearn as sk

In [ ]:
meteo = pd.read_csv("./in-sample//2024_meteo_fcs_comuneimpianto.csv", sep=";")
prod = pd.read_excel(".//in-sample/produzione_2024.xlsx")

In [ ]:
new_df = pd.read_csv("./out-of-sample_regressors/012025_meteo_fcs_comuneimpianto.csv", sep=";")
new_prod = pd.read_excel("./Produzione_impianto.xlsx", sheet_name="produzione_01012025_31012025")

In [6]:
new_df.head()

,Dataora;WDS;WDD;RNF;RHM
0,2025-01-01 00:00:00;7;237.2;0;80.4
1,2025-01-01 01:00:00;7;235.8;0;80
2,2025-01-01 02:00:00;6.8;235.8;0;79.5
3,2025-01-01 03:00:00;6.9;236.7;0;78.8
4,2025-01-01 04:00:00;6.5;237.9;0;78.6


In [37]:
meteo.Dataora = pd.to_datetime(meteo.Dataora)
meteo.Dataora = (meteo.Dataora - meteo.Dataora.min())  / np.timedelta64(1,'D')
new_df.Dataora = pd.to_datetime(new_df.Dataora)
new_df.Dataora = (new_df.Dataora - new_df.Dataora.min())  / np.timedelta64(1,'D')

In [38]:
meteo2 = meteo.iloc[:, 1:]
meteo2.head()

<bound method NDFrame.head of        WDS    WDD  RNF   RHM
0     16.8  204.3  0.0  82.6
1     16.5  200.6  0.0  82.5
2     15.2  199.0  0.0  82.7
3     13.3  202.8  0.0  83.4
4     11.7  198.6  0.0  84.5
...    ...    ...  ...   ...
8779   5.9  237.4  0.0  82.1
8780   6.4  238.6  0.0  83.3
8781   6.7  238.0  0.0  83.4
8782   7.0  236.8  0.0  83.6
8783   7.2  235.9  0.0  83.7

[8784 rows x 4 columns]>

In [39]:
removed = prod[prod.Mwh.isnull()].index
prod.drop(index=removed, inplace=True)
meteo.drop(index=removed, inplace=True)
meteo2.drop(index=removed, inplace=True)

In [21]:
model = sk.ensemble.HistGradientBoostingRegressor(loss="absolute_error", l2_regularization = 0.1, learning_rate = 0.05).fit(y=prod.Mwh[0:7000], X=meteo2.iloc[0:7000,:])

In [22]:
pred = model.predict(X=meteo2.iloc[7000:])

In [23]:
sum(abs(pred - new_prod.Mwh[7000:]))/sum(prod.Mwh[7000:])

0.3810533319529104

In [219]:
lr = [0.1, 0.075, 0.05, 0.025, 0.01, 0.005, 0.001]
l2 = [0, 0.001, 0.005, 0.1, 0.25, 0.3]

for lear in lr:
    for ll in l2:
        model = sk.ensemble.HistGradientBoostingRegressor(loss="absolute_error", l2_regularization = ll, learning_rate = lear).fit(y=prod.Mwh[0:7000], X=meteo2.iloc[0:7000,:])
        pred = model.predict(X=meteo2.iloc[7000:])
        err = sum(abs(pred - prod.Mwh[7000:]))/sum(prod.Mwh[7000:])
        print(lear, ll, err)
        

0.1 0 0.3914669604116107
0.1 0.001 0.3893255733627161
0.1 0.005 0.3913832654830029
0.1 0.1 0.3907403476520195
0.1 0.25 0.3907342873777805
0.1 0.3 0.38679545431150825
0.075 0 0.3873361552894442
0.075 0.001 0.38651012741995516
0.075 0.005 0.38772313995872576
0.075 0.1 0.3866254846858518
0.075 0.25 0.3910140233862019
0.075 0.3 0.3864464509445579
0.05 0 0.3826224737499999
0.05 0.001 0.3826224737499999
0.05 0.005 0.3824779754072146
0.05 0.1 0.3810533319529104
0.05 0.25 0.383598151230852
0.05 0.3 0.38471634436724467
0.025 0 0.4033669788862745
0.025 0.001 0.40322147020269516
0.025 0.005 0.40322147020269516
0.025 0.1 0.4025568861727867
0.025 0.25 0.40144235216057506
0.025 0.3 0.40235138450142194
0.01 0 0.5384388369769383
0.01 0.001 0.5383737094757661
0.01 0.005 0.5384954946732785
0.01 0.1 0.5381919104811066
0.01 0.25 0.5378293162279115
0.01 0.3 0.5382557934858416
0.005 0 0.6847542105995474
0.005 0.001 0.6847542105995474
0.005 0.005 0.6847542105995474
0.005 0.1 0.6851603686082199
0.005 0.25 0.6

In [52]:
model_test = sk.ensemble.HistGradientBoostingRegressor(loss="absolute_error", l2_regularization = 0.1, learning_rate = 0.05).fit(y=prod.Mwh, X=meteo)

In [41]:
new_df2 = new_df.iloc[:,1:]

In [53]:
pred_test = model_test.predict(X=new_df)

In [54]:
sum(abs(pred_test - new_prod.Mwh))/sum(new_prod.Mwh)

0.5508537805835763

## Augmented version

In [ ]:
aug_df = pd.read_csv("./augmented_dataset.csv")
aug_df_test = pd.read_csv("/augmented_dataset.csv")
aug_df.Dataora = pd.to_datetime(aug_df.Dataora)
aug_df_test.Dataora = pd.to_datetime(aug_df_test.Dataora)

In [62]:
aug_df_test.head()

,Dataora,wds_comuneimpianto,wdd_comuneimpianto,rnf_comuneimpianto,rhm_comuneimpianto,wds_comunelimitrofo1,wdd_comunelimitrofo1,rnf_comunelimitrofo1,rhm_comunelimitrofo1,wds_comunelimitrofo2,...,media_wds3_tutti,std_wds3_tutti,media_u_tutti,std_u_tutti,media_v_tutti,std_v_tutti,media_rnf_tutti,std_rnf_tutti,media_rhm_tutti,std_rhm_tutti
0,2025-01-01 00:00:00,7.0,237.2,0.0,80.4,8.8,233.5,0.0,76.9,8.8,...,548.367878,159.578000,-4.782958,0.568275,-6.922163,0.766997,0.0,0.0,80.82,2.573325
1,2025-01-01 01:00:00,7.0,235.8,0.0,80.0,8.9,233.1,0.0,75.7,9.1,...,597.048611,177.228827,-4.944730,0.573290,-7.076845,0.919907,0.0,0.0,80.30,2.933428
2,2025-01-01 02:00:00,6.8,235.8,0.0,79.5,8.5,233.2,0.0,74.1,8.7,...,601.559789,176.713818,-4.804956,0.554989,-6.807218,0.947215,0.0,0.0,79.28,3.266803
3,2025-01-01 03:00:00,6.9,236.7,0.0,78.8,8.6,231.9,0.0,73.3,8.6,...,597.938722,173.422065,-4.848669,0.607695,-6.646031,0.864030,0.0,0.0,78.66,3.357529
4,2025-01-01 04:00:00,6.5,237.9,0.0,78.6,8.1,230.4,0.0,72.6,8.3,...,582.638489,182.528582,-4.769493,0.745933,-6.373543,0.929217,0.0,0.0,78.34,3.562022


In [63]:
aug_df = aug_df.loc[:,['Dataora', 'cos_giorno', 'cos_mese', 'cos_ora','rhm_comunelimitrofo3','rnf_comunelimitrofo3','sin_giorno','sin_mese','sin_ora','std_rhm_tutti','std_u_tutti','std_v_tutti','u_comunelimitrofo4','v_comuneimpianto','wdd_comuneimpianto','wds_comuneimpianto_rol3']]
aug_df_test = aug_df_test.loc[:,['Dataora', 'cos_giorno', 'cos_mese', 'cos_ora','rhm_comunelimitrofo3','rnf_comunelimitrofo3','sin_giorno','sin_mese','sin_ora','std_rhm_tutti','std_u_tutti','std_v_tutti','u_comunelimitrofo4','v_comuneimpianto','wdd_comuneimpianto','wds_comuneimpianto_rol3']]

In [64]:
aug_df.Dataora = (aug_df.Dataora - aug_df.Dataora.min())  / np.timedelta64(1,'D')
aug_df.drop(index=removed, inplace=True)
aug_df_test.Dataora = (aug_df_test.Dataora - aug_df_test.Dataora.min())  / np.timedelta64(1,'D')

In [66]:
aug_df_test.shape

(744, 16)

In [249]:
model2 = sk.ensemble.HistGradientBoostingRegressor(loss="absolute_error", l2_regularization = 0.1, learning_rate = 0.05).fit(y=prod.Mwh[0:7000], X=new_df.iloc[0:7000,:])

In [264]:
pred = model2.predict(X=new_df)

In [265]:
pred2 = pd.DataFrame(pred)
pred2.to_csv("C:/Users/Tommaso/PycharmProjects/hackathon/Data4Econ_Hackaton/output.csv")

In [251]:
sum(abs(pred - prod.Mwh[7000:]))/sum(prod.Mwh[7000:])

0.40966475312997236

In [71]:
aug_df2 = aug_df.iloc[:,1:]
aug_df_test2 = aug_df_test.iloc[:,1:]

In [72]:
model2_test = sk.ensemble.HistGradientBoostingRegressor(loss="absolute_error", l2_regularization = 0.1, learning_rate = 0.05).fit(y=prod.Mwh, X=aug_df2)

In [73]:
pred2_test = model2_test.predict(X=aug_df_test2)

In [74]:
sum(abs(pred_test - new_prod.Mwh))/sum(new_prod.Mwh)

0.5508537805835763

In [246]:
sum(abs(pred - prod.Mwh[7000:])) / sum(prod.Mwh[7000:])
lr = [0.15, 0.1, 0.075, 0.05, 0.025, 0.01]
l2 = [0, 0.0005, 0.001, 0.005, 0.1]

for lear in lr:
    for ll in l2:
        model2 = sk.ensemble.HistGradientBoostingRegressor(loss="absolute_error", l2_regularization=ll,
                                                          learning_rate=lear).fit(y=prod.Mwh[0:7000],
                                                                                  X=new_df.iloc[0:7000, :])
        pred = model2.predict(X=new_df.iloc[7000:])
        err = sum(abs(pred - prod.Mwh[7000:])) / sum(prod.Mwh[7000:])
        print(lear, ll, err)

0.15 0 0.4041286864968295
0.15 0.0005 0.4051694565618736
0.15 0.001 0.4046349517253326
0.15 0.005 0.4070026164813169
0.15 0.1 0.4209631794843724
0.1 0 0.39039328255363553
0.1 0.0005 0.39006181293439424
0.1 0.001 0.3894718871710709
0.1 0.005 0.392794834921443
0.1 0.1 0.39465973145529065
0.075 0 0.41697362780780145
0.075 0.0005 0.41816378300983204
0.075 0.001 0.4170324816763459
0.075 0.005 0.4181660452963478
0.075 0.1 0.4169750503098995
0.05 0 0.4231511364152062
0.05 0.0005 0.4234279650097322
0.05 0.001 0.42324496699496034
0.05 0.005 0.42175620933986235
0.05 0.1 0.40966475312997236
0.025 0 0.43825455594443674
0.025 0.0005 0.4496620811785256
0.025 0.001 0.44892160251379737
0.025 0.005 0.4499590366591419
0.025 0.1 0.4491066031842389
0.01 0 0.6044308329596559
0.01 0.0005 0.6044308329596559
0.01 0.001 0.6044308329596559
0.01 0.005 0.6044308330528053
0.01 0.1 0.6081291478941666
